# Normalização e recorte temporal
Parto do consolidado atual, examino datas e DOIs, aplico as cinco decisões bibliográficas
e seleciono publicações de 2020 a 2022. Datas anuais continuam sem mês ou dia inventado.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))

from src import config as cfg
from src.dados import normalize_bibliography, resolve_doi_conflicts, filter_period

pd.set_option("display.max_colwidth", 120)
from src.io_utils import atomic_json, atomic_parquet
cfg.ensure_directories()


## Inspecionar o consolidado e normalizar os campos

`normalize_bibliography()` cria DOI normalizado, data interpretada, ano, precisão e validade do DOI. O texto original permanece disponível para comparação. A transformação não inventa mês ou dia para datas anuais.

In [2]:
consolidado = pd.read_parquet(cfg.STAGED / "03_consolidado.parquet")
normalizado = normalize_bibliography(consolidado)
display(Markdown(f"**Artigos consolidados:** {len(consolidado):,}"))
display(normalizado[["cord_uid", "doi", "doi_normalizado", "publish_time", "data_publicacao", "ano_publicacao", "precisao_data"]].head(5))
display(normalizado["doi_valido"].value_counts().rename("Artigos").to_frame())

**Artigos consolidados:** 581,015

,cord_uid,doi,doi_normalizado,publish_time,data_publicacao,ano_publicacao,precisao_data
0,ug7v899j,10.1186/1471-2334-1-6,10.1186/1471-2334-1-6,2001-07-04,2001-07-04,2001,dia
1,02tnwd4m,10.1186/rr14,10.1186/rr14,2000-08-15,2000-08-15,2000,dia
2,ejv2xln0,10.1186/rr19,10.1186/rr19,2000-08-25,2000-08-25,2000,dia
3,2b73a28n,10.1186/rr44,10.1186/rr44,2001-02-22,2001-02-22,2001,dia
4,9785vg6d,10.1186/rr61,10.1186/rr61,2001-05-11,2001-05-11,2001,dia


,Artigos
doi_valido,
True,580996
False,19


## Examinar a precisão das datas

Exibo contagens de datas diárias, anuais e inválidas e exemplos da interpretação. Datas explícitas posteriores ao release são preservadas pela regra de datas; a presença delas não implica cobertura completa do ano de 2022.

In [3]:
display(normalizado["precisao_data"].value_counts().rename("Artigos").to_frame())
anuais = normalizado.loc[normalizado["precisao_data"].eq("ano")]
assert anuais["data_publicacao"].isna().all()
display(anuais[["cord_uid", "publish_time", "data_publicacao", "ano_publicacao"]].head(5))
posteriores = normalizado.loc[normalizado["data_publicacao"].gt(pd.Timestamp(cfg.RELEASE))]
display(Markdown(f"Há **{len(posteriores):,} registros com data explícita posterior ao release**, mantidos conforme a regra preservada."))
display(posteriores[["cord_uid", "publish_time", "data_publicacao"]].sort_values("data_publicacao").head(5))

,Artigos
precisao_data,
dia,436633
ano,144382


,cord_uid,publish_time,data_publicacao,ano_publicacao
239,g4puurhk,2008,NaT,2008
240,ke0tkpso,2008,NaT,2008
247,t579ysgl,2009,NaT,2009
345,25v5mr3c,2009,NaT,2009
365,se4unhi8,2009,NaT,2009


Há **327 registros com data explícita posterior ao release**, mantidos conforme a regra preservada.

,cord_uid,publish_time,data_publicacao
489845,1m4tl6gn,2022-06-06,2022-06-06
308802,hedmmk5j,2022-06-06,2022-06-06
528629,8r5mwsus,2022-06-08,2022-06-08
412888,b28ie7dt,2022-06-10,2022-06-10
264623,546lnqji,2022-06-10,2022-06-10


## Mostrar as decisões dos conflitos de DOI

Carrego as cinco decisões existentes e observo os registros afetados. `resolve_doi_conflicts()` valida os representantes e aplica a mesma regra usada pela preparação oficial. Não escolho representantes a partir dos exemplos exibidos.

In [4]:
resolucoes = pd.read_csv(cfg.DOCS / "resolucoes_doi.csv")
display(resolucoes)
display(normalizado.loc[normalizado["doi_normalizado"].isin(resolucoes["doi_normalizado"]),
    ["cord_uid", "doi_normalizado", "title"]].sort_values(["doi_normalizado", "cord_uid"]))
canonico = resolve_doi_conflicts(normalizado, resolucoes)
display(pd.Series({"Antes das resoluções": len(normalizado), "Após resoluções": len(canonico),
    "Registros removidos pelas decisões": len(normalizado) - len(canonico)}, name="Artigos").to_frame())

,doi_normalizado,cord_uid,motivo
0,10.12200/j.issn.1003-0034.2022.04.012,a4n6awcv,Titulo e data concordam com OpenAlex W4225125286; outro titulo nao recebe estas citacoes
1,10.12688/f1000research.75628.2,0v09cwir,Representante sem marcador preprint; OpenAlex W2742590913
2,10.12688/f1000research.76032.2,mz3u7d29,Representante sem marcador preprint; OpenAlex W4224292727
3,10.12688/hrbopenres.13461.2,9mnzg2zl,Mesmo titulo e DOI; OpenAlex W4229058819
4,10.17533/udea.iee.v40n1e08,ui6cpgfv,Mesmo titulo e DOI; preservar data explicita original e sinalizar divergencia com OpenAlex W4226071422


,cord_uid,doi_normalizado,title
87681,a4n6awcv,10.12200/j.issn.1003-0034.2022.04.012,[Effect of preoperative waiting time on prognosis of elderly patients with hip fracture].
385083,xvzo2m9j,10.12200/j.issn.1003-0034.2022.04.012,[Treatment of abduction and insertion femoral neck fracture with closed reduction technique]
92489,0v09cwir,10.12688/f1000research.75628.2,Tourette syndrome research highlights from 2020.
175877,7qt23pfs,10.12688/f1000research.75628.2,Tourette syndrome research highlights from 2020 (preprint)
181076,d7ty01d8,10.12688/f1000research.76032.2,"Assessment of fear, anxiety, obsession and functional impairment due to COVID-19 amongst health-care workers and tra..."
98086,mz3u7d29,10.12688/f1000research.76032.2,"Assessment of fear, anxiety, obsession and functional impairment due to COVID-19 amongst health-care workers and tra..."
143879,4abjix02,10.12688/hrbopenres.13461.2,"Protocol for an exploratory, longitudinal single case study of a novel palliative care rehabilitative service"
57671,9mnzg2zl,10.12688/hrbopenres.13461.2,"Protocol for an exploratory, longitudinal single case study of a novel palliative care rehabilitative service."
192079,r89x2m5v,10.17533/udea.iee.v40n1e08,"Characteristics, treatment, and nursing care of patients infected by Sars-CoV-2 hospitalized in intensive care units..."
110030,ui6cpgfv,10.17533/udea.iee.v40n1e08,"Characteristics, treatment, and nursing care of patients infected by Sars-CoV-2 hospitalized in intensive care units..."


,Artigos
Antes das resoluções,581015
Após resoluções,581010
Registros removidos pelas decisões,5


## Aplicar o recorte de 2020 a 2022
O resultado é gravado como corpus do período da execução atual.
Não há lista de referência histórica a restaurar ou comparar.

In [5]:
periodo = filter_period(canonico)
atomic_parquet(cfg.STAGED / "04_corpus_periodo.parquet", periodo)
display(pd.Series({"Após resoluções": len(canonico), "Corpus do período": len(periodo),
                   "Fora do recorte ou sem ano válido": len(canonico) - len(periodo)}, name="Artigos").to_frame())

,Artigos
Após resoluções,581010
Corpus do período,507068
Fora do recorte ou sem ano válido,73942


## Explorar a cobertura anual

Agrupo por ano e precisão da data. O gráfico inclui os registros que informam somente o ano, sem tratá-los como janeiro. A distribuição de 2022 corresponde ao release de junho e não permite uma comparação de anos com cobertura uniforme.

In [6]:
cobertura = periodo.groupby(["ano_publicacao", "precisao_data"]).size().unstack(fill_value=0)
cobertura["Total"] = cobertura.sum(axis=1)
display(cobertura)

precisao_data,ano,dia,Total
ano_publicacao,,,
2020,16071,130268,146339
2021,64029,189698,253727
2022,31937,75065,107002


## Gravar a cobertura e seguir para os textos
A cobertura anual documenta as datas disponíveis. No notebook 04, título e resumo
serão lematizados, e os IDs dos textos utilizáveis serão gerados a partir desta curadoria.
O release de junho não representa toda a produção de 2022.

In [7]:
cobertura.to_csv(cfg.OUTPUTS / "tables/cobertura_periodo.csv")
display(Markdown("Corpus e cobertura atuais gravados. Continue no notebook 04."))

Corpus e cobertura atuais gravados. Continue no notebook 04.